In [ ]:
import os
import warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import spearmanr, kruskal
from sklearn.compose import TransformedTargetRegressor
from sklearn.dummy import DummyRegressor
from sklearn.feature_selection import SelectKBest, f_regression
from sklearn.impute import SimpleImputer
from sklearn.inspection import permutation_importance, PartialDependenceDisplay
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split, GridSearchCV, KFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.svm import SVR

sns.set_theme(style='whitegrid')
pd.set_option('display.width', 220)
pd.set_option('display.max_columns', 30)

DATA_PATH = 'deforestation_dataset.csv'
RANDOM_STATE = 42
TARGET_NAME = 'Forest_Loss_Area_km2'
ALPHA = 0.05

if not os.path.exists(DATA_PATH):
    try:
        from google.colab import files
        uploaded = files.upload()
    except Exception:
        pass

if not os.path.exists(DATA_PATH):
    raise FileNotFoundError('deforestation_dataset.csv not found. Upload it next to this notebook.')

print('PHASE 1: DATA PREPROCESSING')
df = pd.read_csv(DATA_PATH)
df.columns = df.columns.str.strip().str.replace(' ', '_')
print('Initial Dataset Shape:', df.shape)
print('Dataset Columns:', df.columns.tolist())
print('First 5 Rows:')
print(df.head())
print('Missing Values Before Cleaning:')
print(df.isnull().sum())
print('Duplicate Rows:', df.duplicated().sum())
print('Descriptive Statistics:')
print(df.describe().T)
df = df.drop_duplicates().reset_index(drop=True)


def find_col(frame, *keys):
    for key in keys:
        parts = key if isinstance(key, tuple) else (key,)
        for c in frame.columns:
            if all(p in c.lower() for p in parts):
                return c
    return None


country_col = find_col(df, 'country')
year_col = find_col(df, 'year')
policy_col = find_col(df, 'policy')
corruption_col = find_col(df, 'corruption')
illegal_col = find_col(df, 'illegal')
gdp_col = find_col(df, 'gdp')
population_col = find_col(df, 'population')
rainfall_col = find_col(df, 'rain')

target_col = TARGET_NAME if TARGET_NAME in df.columns else find_col(df, 'forest_loss', 'tree_cover_loss')
if target_col is None:
    raise ValueError('No deforestation target column found. Available: ' + str(list(df.columns)))
other_targets = [c for c in df.columns if c != target_col and ('forest_loss' in c.lower() or 'tree_cover_loss' in c.lower())]
print('Target Variable:', target_col)
print('Excluded from features (another measure of the same outcome):', other_targets)

ordinal_map = {'very low': 0, 'low': 1, 'medium': 2, 'moderate': 2, 'high': 3, 'strict': 4, 'very high': 4, 'very strict': 5}
for c in [policy_col, corruption_col]:
    if c is not None and df[c].dtype == object:
        mapped = df[c].astype(str).str.strip().str.lower().map(ordinal_map)
        if mapped.notna().all():
            df[c] = mapped
        else:
            df[c] = LabelEncoder().fit_transform(df[c].astype(str))
        print('Encoded categorical column:', c)

df['Country_Encoded'] = LabelEncoder().fit_transform(df[country_col].astype(str))
other_cat = [c for c in df.select_dtypes(exclude=[np.number]).columns if c != country_col]
if other_cat:
    df = pd.get_dummies(df, columns=other_cat, dtype=float)
    print('One-hot encoded columns:', other_cat)

feature_cols = [c for c in df.select_dtypes(include=[np.number]).columns
                if c != target_col and c not in other_targets and c != 'Country_Encoded']
print('Country is kept for reporting only (label codes have no numeric meaning, so it is not a model input).')
print('Selected Feature Columns:', feature_cols)

X = df[feature_cols].copy()
y = df[target_col].copy()
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=RANDOM_STATE, shuffle=True)
print('Training Set Shape:', X_train.shape)
print('Testing Set Shape:', X_test.shape)
print('Imputation and standardization are fitted inside each model pipeline on training data only.')


def make_model(**svr_params):
    return TransformedTargetRegressor(
        regressor=Pipeline([
            ('impute', SimpleImputer(strategy='median')),
            ('scale', StandardScaler()),
            ('svr', SVR(**svr_params))
        ]),
        transformer=StandardScaler()
    )


def evaluate(name, y_true, y_pred):
    mse = mean_squared_error(y_true, y_pred)
    return {'Model': name, 'MAE': round(mean_absolute_error(y_true, y_pred), 4), 'MSE': round(mse, 4),
            'RMSE': round(float(np.sqrt(mse)), 4), 'R2_Score': round(r2_score(y_true, y_pred), 4)}


kf5 = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

print('PHASE 2: MODEL BUILDING AND EVALUATION')
raw_linear = Pipeline([('impute', SimpleImputer(strategy='median')), ('scale', StandardScaler()),
                       ('svr', SVR(kernel='linear', C=1.0))])
raw_linear.fit(X_train, y_train)

svr_linear = make_model(kernel='linear', C=1.0)
svr_linear.fit(X_train, y_train)

print('Initial Linear SVR (scaled target) - Training and Testing Performance:')
print(pd.DataFrame([
    evaluate('Linear SVR (Train)', y_train, svr_linear.predict(X_train)),
    evaluate('Linear SVR (Test)', y_test, svr_linear.predict(X_test))
]).to_string(index=False))

print('Why the target is scaled: SVR with an unscaled target of this size cannot fit it with C=1.')
print(pd.DataFrame([
    evaluate('Linear SVR, raw target (Train)', y_train, raw_linear.predict(X_train)),
    evaluate('Linear SVR, scaled target (Train)', y_train, svr_linear.predict(X_train))
]).to_string(index=False))
print('Spread of raw-target predictions (std):', round(float(np.std(raw_linear.predict(X_train))), 2),
      'vs spread of the actual target (std):', round(float(np.std(y_train)), 2))

dummy = DummyRegressor(strategy='mean').fit(X_train, y_train)
svr_poly = make_model(kernel='poly', degree=2, C=1.0).fit(X_train, y_train)
svr_rbf = make_model(kernel='rbf', C=1.0, gamma='scale').fit(X_train, y_train)

param_grid = [
    {'regressor__svr__kernel': ['linear'], 'regressor__svr__C': [0.1, 1.0, 10.0, 100.0],
     'regressor__svr__epsilon': [0.05, 0.1, 0.2]},
    {'regressor__svr__kernel': ['rbf'], 'regressor__svr__C': [0.1, 1.0, 10.0, 100.0],
     'regressor__svr__gamma': ['scale', 'auto', 0.01, 0.1], 'regressor__svr__epsilon': [0.05, 0.1, 0.2]},
    {'regressor__svr__kernel': ['poly'], 'regressor__svr__C': [0.1, 1.0, 10.0],
     'regressor__svr__degree': [2, 3], 'regressor__svr__gamma': ['scale', 0.1], 'regressor__svr__epsilon': [0.1]}
]
grid_search = GridSearchCV(make_model(), param_grid, cv=kf5, scoring='r2', n_jobs=-1)
grid_search.fit(X_train, y_train)
best_model = grid_search.best_estimator_
print('Best Hyperparameters from GridSearchCV:', {k.replace('regressor__svr__', ''): v for k, v in grid_search.best_params_.items()})
print(f'Best 5-Fold CV R2 on the training set: {grid_search.best_score_:.4f}')

models = {'Mean baseline': dummy, 'Linear SVR': svr_linear, 'Polynomial SVR': svr_poly, 'RBF SVR': svr_rbf, 'Tuned SVR': best_model}
comparison_table = pd.DataFrame([evaluate(n + ' (Test)', y_test, m.predict(X_test)) for n, m in models.items()])
comparison_table['CV_R2_Train_Mean'] = [round(cross_val_score(m, X_train, y_train, cv=kf5, scoring='r2').mean(), 4)
                                        for n, m in models.items()]
print('Model Comparison Across Kernels:')
print(comparison_table.to_string(index=False))
print('Tuned SVR on training data:')
print(pd.DataFrame([evaluate('Tuned SVR (Train)', y_train, best_model.predict(X_train))]).to_string(index=False))

print('Nested Cross-Validation (tuning repeated inside every outer fold, whole dataset):')
nested_search = GridSearchCV(make_model(), param_grid, cv=KFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE),
                             scoring='r2', n_jobs=-1)
nested_scores = cross_val_score(nested_search, X, y, cv=kf5, scoring='r2')
dummy_scores = cross_val_score(DummyRegressor(strategy='mean'), X, y, cv=kf5, scoring='r2')
print('Tuned SVR nested CV R2 per fold:', np.round(nested_scores, 3))
print(f'Tuned SVR nested CV R2: {nested_scores.mean():.4f} (+/- {nested_scores.std():.4f})')
print(f'Mean baseline CV R2   : {dummy_scores.mean():.4f} (+/- {dummy_scores.std():.4f})')

print('FEATURE SELECTION CHECK (selection done inside each CV fold):')
selection_rows = []
for k in sorted(set([min(3, len(feature_cols)), min(5, len(feature_cols)), len(feature_cols)])):
    sel_model = TransformedTargetRegressor(
        regressor=Pipeline([('impute', SimpleImputer(strategy='median')), ('scale', StandardScaler()),
                            ('select', SelectKBest(f_regression, k=k)), ('svr', SVR(kernel='rbf', C=1.0))]),
        transformer=StandardScaler())
    sc = cross_val_score(sel_model, X, y, cv=kf5, scoring='r2')
    selection_rows.append({'Features_Kept': k, 'CV_R2_Mean': round(sc.mean(), 4), 'CV_R2_Std': round(sc.std(), 4)})
selection_df = pd.DataFrame(selection_rows)
print(selection_df.to_string(index=False))
print('Feature selection helps only if a smaller set scores clearly higher than keeping all features.')

print('PHASE 3: FEATURE ANALYSIS AND INTERPRETATION')
perm_imp = permutation_importance(best_model, X_test, y_test, n_repeats=30, random_state=RANDOM_STATE, scoring='r2')
linear_coefs = svr_linear.regressor_.named_steps['svr'].coef_[0]

rho_list, p_list = [], []
for f in feature_cols:
    rho, p = spearmanr(df[f], df[target_col])
    rho_list.append(rho)
    p_list.append(p)


def bh_adjust(p):
    p = np.asarray(p, dtype=float)
    m = len(p)
    order = np.argsort(p)
    ranked = p[order] * m / (np.arange(m) + 1)
    q = np.minimum.accumulate(ranked[::-1])[::-1]
    out = np.empty(m)
    out[order] = np.minimum(q, 1.0)
    return out


q_list = bh_adjust(p_list)
importance_df = pd.DataFrame({
    'Feature': feature_cols,
    'Spearman_rho': np.round(rho_list, 3),
    'p_value': np.round(p_list, 4),
    'q_value': np.round(q_list, 4),
    'Linear_SVR_Weight': np.round(linear_coefs, 3),
    'Permutation_Importance': np.round(perm_imp.importances_mean, 4),
    'Permutation_Std': np.round(perm_imp.importances_std, 4)
})
importance_df['Effect'] = np.where(importance_df['q_value'] < ALPHA,
                                   np.where(importance_df['Spearman_rho'] > 0, 'raises loss', 'lowers loss'),
                                   'no clear link')
importance_df = importance_df.sort_values('Permutation_Importance', ascending=False).reset_index(drop=True)
print('Feature Importance Analysis:')
print(importance_df.to_string(index=False))
print('Spearman p-values are corrected for testing many features at once (Benjamini-Hochberg q-values).')

print('Feature-by-feature report:')
for label, c in [('GDP', gdp_col), ('Population', population_col), ('Rainfall', rainfall_col),
                 ('Policy strictness', policy_col), ('Corruption', corruption_col), ('Illegal lumbering', illegal_col)]:
    if c is not None and c in importance_df['Feature'].values:
        r = importance_df[importance_df['Feature'] == c].iloc[0]
        print(f"  {label} ({c}): Spearman rho = {r['Spearman_rho']:.2f}, q = {r['q_value']:.3f}, "
              f"linear weight = {r['Linear_SVR_Weight']:.2f}, permutation importance = {r['Permutation_Importance']:.3f} -> {r['Effect']}")

print('Deforestation by group (average of', target_col + '):')
group_frames = {}
if policy_col is not None:
    pol_series = df[policy_col] if df[policy_col].nunique() <= 8 else pd.qcut(df[policy_col], 4, duplicates='drop')
    group_frames['Policy strictness'] = df.groupby(pol_series)[target_col].agg(['mean', 'count'])
if illegal_col is not None:
    group_frames['Illegal lumbering (quartiles)'] = df.groupby(pd.qcut(df[illegal_col], 4, duplicates='drop'))[target_col].agg(['mean', 'count'])
if gdp_col is not None:
    group_frames['GDP (quartiles)'] = df.groupby(pd.qcut(df[gdp_col], 4, duplicates='drop'))[target_col].agg(['mean', 'count'])
for name, g in group_frames.items():
    print(name)
    print(g.round(2).to_string())

country_stats = df.groupby(country_col).agg(
    Records=(target_col, 'size'),
    Avg_Loss=(target_col, 'mean'),
    Avg_Policy=(policy_col, 'mean') if policy_col else (target_col, 'size'),
    Avg_Illegal=(illegal_col, 'mean') if illegal_col else (target_col, 'size'),
    Avg_GDP=(gdp_col, 'mean') if gdp_col else (target_col, 'size')
).sort_values('Avg_Loss', ascending=False)
print('Countries ranked by average', target_col + ':')
print(country_stats.round(2).head(10).to_string())
country_groups = [g[target_col].values for _, g in df.groupby(country_col) if len(g) > 1]
if len(country_groups) > 2:
    H, p_country = kruskal(*country_groups)
    print(f'Kruskal-Wallis (does average loss differ between countries?): H = {H:.2f}, p = {p_country:.3f}')
else:
    p_country = np.nan

print('PHASE 4: VISUALIZATIONS')
plt.figure(figsize=(11, 8))
sns.heatmap(df[feature_cols + [target_col]].corr(method='spearman'), annot=True, fmt='.2f', cmap='coolwarm', center=0, linewidths=0.5)
plt.title('Spearman Correlation Heatmap (features and deforestation)')
plt.tight_layout()
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(15, 5.5))
imp_plot = importance_df.sort_values('Permutation_Importance')
axes[0].barh(imp_plot['Feature'], imp_plot['Permutation_Importance'], xerr=imp_plot['Permutation_Std'], color='teal', alpha=0.85)
axes[0].axvline(0, color='black', lw=1)
axes[0].set_title('Permutation importance (tuned SVR, test set)')
axes[0].set_xlabel('Drop in R2 when the feature is shuffled')
axes[1].scatter(y_test, best_model.predict(X_test), color='teal', edgecolors='k', s=70, alpha=0.8)
lims = [min(y_test.min(), best_model.predict(X_test).min()), max(y_test.max(), best_model.predict(X_test).max())]
axes[1].plot(lims, lims, 'r--', lw=2, label='Ideal fit')
axes[1].set_title('Tuned SVR: actual vs predicted (test set)')
axes[1].set_xlabel('Actual ' + target_col)
axes[1].set_ylabel('Predicted ' + target_col)
axes[1].legend()
plt.tight_layout()
plt.show()

plt.figure(figsize=(8, 4.5))
sns.barplot(data=comparison_table, x='Model', y='R2_Score', palette='Set2')
plt.axhline(0, color='black', lw=1)
plt.title('Kernel comparison: test R2 (0 = predicting the average)')
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()

top_corr = importance_df.reindex(importance_df['Spearman_rho'].abs().sort_values(ascending=False).index).head(3)['Feature'].tolist()
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
for ax, f in zip(axes, top_corr):
    sns.regplot(data=df, x=f, y=target_col, ax=ax, scatter_kws={'alpha': 0.6, 'color': 'royalblue'}, line_kws={'color': 'crimson', 'lw': 2})
    r = importance_df[importance_df['Feature'] == f].iloc[0]
    ax.set_title(f"{f}\nrho = {r['Spearman_rho']:.2f}, q = {r['q_value']:.2f}")
plt.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(15, 4.5))
PartialDependenceDisplay.from_estimator(best_model, X_train, features=top_corr, ax=ax)
fig.suptitle('Partial dependence of the tuned SVR on the three features most correlated with deforestation')
plt.tight_layout()
plt.show()

if group_frames:
    fig, axes = plt.subplots(1, len(group_frames), figsize=(5.5 * len(group_frames), 4.5))
    axes = np.atleast_1d(axes)
    for ax, (name, g) in zip(axes, group_frames.items()):
        ax.bar([str(i) for i in g.index], g['mean'], color='seagreen')
        ax.set_title(name)
        ax.set_ylabel('Average ' + target_col)
        ax.tick_params(axis='x', rotation=30)
    plt.tight_layout()
    plt.show()

plt.figure(figsize=(9, 5))
top_countries = country_stats.head(10).iloc[::-1]
plt.barh(top_countries.index.astype(str), top_countries['Avg_Loss'], color='darkorange')
plt.title('Countries with the highest average ' + target_col)
plt.xlabel('Average ' + target_col)
plt.tight_layout()
plt.show()

print('Decision boundaries do not apply to SVM regression; feature importance, partial dependence and actual-vs-predicted plots are used instead.')

print('PHASE 4: REPORT AND RECOMMENDATIONS')
tuned_test = comparison_table[comparison_table['Model'] == 'Tuned SVR (Test)'].iloc[0]
has_skill = bool(nested_scores.mean() > dummy_scores.mean() + 0.05 and nested_scores.mean() > 0.10)
sig_features = importance_df[importance_df['q_value'] < ALPHA]
print('SUMMARY (generated from this run):')
print(f"1. Tuned SVR test R2 = {tuned_test['R2_Score']:.3f}, RMSE = {tuned_test['RMSE']:.1f}; nested CV R2 = {nested_scores.mean():.3f} "
      f"vs {dummy_scores.mean():.3f} for always predicting the average.")
if has_skill:
    print('   The SVM explains a meaningful share of deforestation beyond the average, so its feature importances are informative.')
else:
    print('   The SVM does not predict deforestation better than the average. Its feature importances and kernel comparison')
    print('   are therefore not reliable evidence about what drives deforestation in this dataset.')
if len(sig_features):
    print('2. Features with a statistically significant link to deforestation (after correction):')
    for _, r in sig_features.iterrows():
        print(f"   {r['Feature']}: {r['Effect']} (rho = {r['Spearman_rho']:.2f}, q = {r['q_value']:.3f})")
else:
    print('2. No feature has a statistically significant link to deforestation after correcting for multiple tests.')
if not np.isnan(p_country):
    print(f"3. Average loss differs between countries: Kruskal-Wallis p = {p_country:.3f} -> "
          + ('significant' if p_country < ALPHA else 'not significant, so country rankings may reflect chance')
          + f" (only about {int(country_stats['Records'].median())} records per country, so treat rankings with caution).")
top3 = country_stats.head(3)
print('4. Highest average loss: ' + ', '.join(f"{c} ({v:,.0f})" for c, v in zip(top3.index, top3['Avg_Loss'])) + '.')
print()
print('Recommendations (hypotheses to test, not proven effects):')
if has_skill and len(sig_features):
    for _, r in sig_features.iterrows():
        if r['Effect'] == 'raises loss':
            print(f"  - {r['Feature']} is associated with more deforestation: prioritize interventions that reduce or regulate it.")
        else:
            print(f"  - {r['Feature']} is associated with less deforestation: consider strengthening or expanding it.")
else:
    print('  - Do not base policy on this model: the data in this file does not show reliable drivers of deforestation.')
    print('  - Collect more records per country and consistent yearly data, plus variables such as enforcement budgets,')
    print('    commodity prices, road building and land tenure.')
print('  - Pilot interventions in the highest-loss countries above (enforcement against illegal lumbering, protected-area')
print('    expansion, support for agricultural alternatives) and measure the change before scaling up.')
print('  - Re-run this analysis with more data; with about 100 records, conclusions are tentative.')
